In [1]:
from datasets import load_dataset

ds = load_dataset("dair-ai/emotion", "split")

In [ ]:
print(ds)

In [ ]:
import pandas as pd
train_df = ds['train'].to_pandas()
print(train_df.head())

In [ ]:

print("--- Null Counts ---")
print(train_df.isnull().sum())
print("\n--- Duplicate Counts ---")
print(train_df.duplicated().sum())

In [ ]:
# Print the count of each emotion label
print("--- Emotion Distribution ---")
print(train_df['label'].value_counts())

In [ ]:
# Calculate the number of words in each text
train_df['words_count'] = train_df['text'].apply(lambda x: len(x.split()))

# Get descriptive statistics on the word count
print("--- Text Length Distribution (Word Count) ---")
print(train_df['words_count'].describe())

In [ ]:
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

ds = load_dataset("dair-ai/emotion", "split")
train_df = ds['train'].to_pandas()
test_df = ds['test'].to_pandas()
X_train = train_df['text']
y_train = train_df['label']
X_test = test_df['text']
y_test = test_df['label']
label_map = {
    0: 'sadness', 1: 'joy', 2: 'love',
    3: 'anger', 4: 'fear', 5: 'surprise'
}
target_names = list(label_map.values())

In [ ]:
text_clf = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),  
        stop_words='english',
        max_features=10000  
    )),
    
    ('clf', LogisticRegression(
        class_weight='balanced', 
        max_iter=500,        
        random_state=42,
        n_jobs=-1            
    ))
])

print("training model on lr")
text_clf.fit(X_train, y_train)
print("training complete")

In [ ]:
y_pred = text_clf.predict(X_test)
print(f"\nOverall Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\n--- Detailed Classification Report ---")
print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import numpy as np
import joblib 

def create_synthetic_data():
    """Creates a small, balanced dataset for demonstration."""
    data = {
        'text': [
            "I am so happy and delighted to see you again!", # joy
            "This makes me feel utterly devastated and low.", # sadness
            "I was furious when the meeting was cancelled without notice.", # anger
            "That sudden noise startled me and gave me a massive fright.", # fear
            "I feel deep affection for my dog; she is my whole world.", # love
            "Wow, I didn't expect that fantastic gift at all!", # surprise
            "My heart aches from the constant worry about the situation.", # sadness
            "This whole situation is just absolutely ridiculous and maddening.", # anger
            "I can't believe how cheerful and bright the weather is today.", # joy
            "I have a terrible feeling of dread and uncertainty about this.", # fear
            "It was a genuine pleasure and I truly adore your kind company.", # love
            "What a twist! I am genuinely shocked by the sudden outcome.", # surprise
            "I feel a wave of intense despair washing over me right now.", # sadness
            "I'm bubbling with excitement and glee, this is amazing!", # joy
            "I was completely taken aback, what an unexpected turn!", # surprise
            "Why did you do that? I am filled with rage and resentment.", # anger
        ],
        'label': [
            'joy', 'sadness', 'anger', 'fear', 'love', 'surprise', 
            'sadness', 'anger', 'joy', 'fear', 'love', 'surprise',
            'sadness', 'joy', 'surprise', 'anger'
        ]
    }
    return pd.DataFrame(data)

def train_emotion_model(df, model_filename="emotion_model.pkl"):
    """Trains a TF-IDF + Logistic Regression pipeline and saves it."""
    X_train, X_test, y_train, y_test = train_test_split(
        df['text'], 
        df['label'], 
        test_size=0.4, 
        random_state=42, 
        stratify=df['label']
    )
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(
            ngram_range=(1, 2), 
            stop_words='english', 
            max_features=1000
        )),
        ('clf', LogisticRegression(
            C=1.0, 
            solver='liblinear', 
            multi_class='ovr', 
            class_weight='balanced', 
            random_state=42, 
            max_iter=500
        ))
    ])

    print("--- Training Model ---")
    pipeline.fit(X_train, y_train)
    joblib.dump(pipeline, model_filename)
    print(f"Model successfully saved to {model_filename}")
    y_pred = pipeline.predict(X_test)
    print("\nSimulated Model Performance (Test Set):")
    print(classification_report(y_test, y_pred, zero_division=0))
    
    return pipeline

def predict_emotion(pipeline, sentence):
    """Predicts the emotion of a single input sentence."""
    prediction = pipeline.predict([sentence])[0]
    probabilities = pipeline.predict_proba([sentence])[0]
    labels = pipeline.classes_
    confidence = np.max(probabilities) * 100
    
    return prediction, confidence

if __name__ == "__main__":
    MODEL_FILE = "emotion_model.pkl"
    
    data_df = create_synthetic_data()
    model_pipeline = train_emotion_model(data_df, model_filename=MODEL_FILE)
    
    print("-" * 50)
    print("Emotion Classification Model is Ready for Testing!")
    print(f"Model trained on classes: {', '.join(model_pipeline.classes_)}")
    print(f"Model saved as: {MODEL_FILE}")
    print("-" * 50)

    while True:
        user_input = input("\nEnter a sentence to classify (or type 'quit' to exit): \n> ").strip()
        
        if user_input.lower() == 'quit':
            print("Exiting the predictor. Goodbye!")
            break
        
        if not user_input:
            print("Please enter a sentence.")
            continue
            
        try:
            emotion, confidence = predict_emotion(model_pipeline, user_input)
            print(f"\n[CLASSIFICATION RESULT]")
            print(f"Predicted Emotion: \033[1m{emotion.upper()}\033[0m")
            print(f"Confidence: {confidence:.2f}%")
            print("-" * 50)
            
        except Exception as e:
            print(f"An error occurred during prediction: {e}")
            print("-" * 50)